In [0]:
# GOLD LAYER — IMPERATIVE APPROACH
# cell 1 Imports and configurations
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
#spark_round  é a função do Spark que arredonda colunas.

#source - silver imperative (ADLS paths)
STORAGE_ACCOUNT = "marketpulsedatalake"
SILVER_FACT_PRICES_PATH = (f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_prices/"
)
#target - gold ( new container)
GOLD_DAILY_SUMMARY_PATH = (f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/daily_summary/"
)
GOLD_VOLUME_ANALYSIS_PATH = (
    f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/volume_analysis/"
)
GOLD_MOVING_AVERAGES_PATH = (
    f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/moving_averages/"
)
GOLD_VOLATILITY_PATH = (
    f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/volatility/"
)
GOLD_STOCK_COMPARISON_PATH = (
    f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/stock_comparison/"
)

print("✅ Configuration loaded")

In [0]:
def read_silver_fact_prices() -> DataFrame:
    """
    Reads the silver layer fact_prices table.
    
    Returns:"""
    return spark.read.format("delta").load(SILVER_FACT_PRICES_PATH)

silver_fact_prices = read_silver_fact_prices()
silver_fact_prices.printSchema()
silver_fact_prices.show(5)    

## 1. gold_daily_summary

Daily metrics per stock and trading day to enable basic performance analysis.

**Columns:**
- `symbol`, `trade_date`, `open`, `high`, `low`, `close`, `volume` — from Silver
- `daily_return_pct` — ((close - open) / open) × 100, rounded to 2 decimals
- `intraday_range` — high - low (absolute price range during the day)
- `intraday_range_pct` — (high - low) / open × 100 (range as % of open)
- `ingest_timestamp` — when this row was written

**Business value:**
- Track daily winners and losers
- Identify volatile vs calm trading days
- Compare intraday volatility across stocks of different price ranges

In [0]:


def build_daily_summary(df_fact: DataFrame) -> DataFrame:
    """
    Builds daily summary metrics per stock and trading day.
    
    Returns:
        DataFrame with OHLCV + return %, range, range %.
    """
    return(
        df_fact
            .withColumn(
                "daily_return_pct",
                F.when(
                    F.col("open") != 0,
                    F.round(((F.col("close") - F.col("open")) / F.col("open"))*100, 2)
                    )
            )
            .withColumn(
                "intraday_range",
                F.round(F.col("high") - F.col("low"),2)
            )
            .withColumn(
                "intraday_range_pct",
                F.when(
                    F.col("open") != 0,
                    F.round(((F.col("high") - F.col("low")) / F.col("open"))*100,2)
                )
                
            )
            .withColumn(
                "ingest_timestamp",
                F.current_timestamp()
            )
     
    )
    



In [0]:
# Test the function
df_fact = read_silver_fact_prices()
df_summary = build_daily_summary(df_fact)
df_summary.printSchema()
df_summary.show(5)

In [0]:
def write_gold_table(
    df: DataFrame,
    path: str,
    table_name: str,
    mode: str = "overwrite",
    partition_by: list = None
) -> None:
    """
    Writes a DataFrame to a Delta table.
    
    Args:
        df: DataFrame to write.
        path: ADLS path to write to.
        tablename: Name of the table.
        mode: Write mode.
        partition_by: Partition column.
    """
    writer = df.write.format("delta").mode(mode)
    if partition_by is not None:
        writer = writer.partitionBy(partition_by)
    writer.save(path)
    print(f" Wrote {table_name} to {path}")

write_gold_table(df_summary, GOLD_DAILY_SUMMARY_PATH, "daily_summary", "overwrite")
    

In [0]:
%sql
SELECT *
FROM delta.`abfss://gold@marketpulsedatalake.dfs.core.windows.net/daily_summary/`
WHERE symbol = 'AAPL'
ORDER BY trade_date DESC

In [0]:
# Confirma onde escreveu
print(f"Path configurado: {GOLD_DAILY_SUMMARY_PATH}")

# Lista o container gold
print("\nConteúdo do container gold:")
files = dbutils.fs.ls("abfss://gold@marketpulsedatalake.dfs.core.windows.net/")
for f in files:
    print(f.path)

## 2. gold_volume_analysis

Volume metrics per stock to identify unusual trading activity.

**Columns:**
- `symbol`, `trade_date`, `volume` — from Silver
- `aavg_volume_30d_millions` — 30-day rolling average of volume per symbol
- `volume_vs_avg_pct` — today's volume as % of the 30-day average
- `high_volume_day` — boolean flag, True when volume exceeds 150% of the average
- `ingest_timestamp` — when this row was written

**Business value:**
- Detect unusual trading activity (potentially news-driven)
- Spot accumulation/distribution patterns
- Identify liquidity for trading strategies

In [0]:
def build_volume_analysis(silver_path: str) -> DataFrame:
    """
    Builds volume analysis metrics per stock and trading day.
    
    Args:
        silver_path: ADLS path to Silver fact_prices Delta table
    
    Returns:
        DataFrame with rolling 30-day average volume and high-volume flags.
    """

    query = f"""
    WITH volume_analysis AS (
        SELECT
            symbol,
            trade_date,
            volume,
            CURRENT_TIMESTAMP() AS ingest_timestamp ,

            AVG(volume) OVER (
                PARTITION BY symbol
                ORDER BY trade_date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ) AS avg_volume_30d

        FROM delta.`{silver_path}`
    )

    SELECT
        symbol,
        trade_date,
        volume,
        ROUND(avg_volume_30d / 1000000, 2)  AS avg_volume_30d_millions,

        ROUND(
            (volume / avg_volume_30d) * 100,
            2
        ) AS volume_vs_avg_pct,

        CASE
            WHEN (volume / avg_volume_30d) * 100 > 150
            THEN TRUE
            ELSE FALSE
        END AS high_volume_day,

        ingest_timestamp

    FROM volume_analysis
    """
    return spark.sql(query)

In [0]:
df_volume = build_volume_analysis(SILVER_FACT_PRICES_PATH)
df_volume.limit(10).display()

## 3. gold_moving_averages

Simple Moving Averages (SMA) per stock — classic trend indicators.

**Columns:**
- `symbol`, `trade_date`, `close` — from Silver
- `sma_7d` — Simple Moving Average over the last 7 trading days
- `sma_30d` — Simple Moving Average over the last 30 trading days
- `sma_signal` — "bullish" if sma_7d > sma_30d, "bearish" otherwise
- `ingest_timestamp` — when this row was written

**Business value:**
- Identify short-term vs long-term price trends
- Detect golden cross (sma_7d crossing above sma_30d) and death cross (opposite)
- Smooth out daily price noise to see the underlying trend

In [0]:
def build_moving_averages(silver_path: str) -> DataFrame:
    """
    Builds Simple Moving Averages (7d and 30d) with bullish/bearish signal.
    
    Returns:
        DataFrame with sma_7d, sma_30d and sma_signal per stock and day.
    """
    query = f"""
    WITH sma_data AS (
        SELECT 
            symbol,
            trade_date,
            close,
            ROUND(AVG(close) over (
                PARTITION BY symbol
                ORDER BY trade_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ), 2) AS sma_7d,
            ROUND(AVG(close) over (
                PARTITION BY symbol
                ORDER BY trade_date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ), 2) AS sma_30d,
            CURRENT_TIMESTAMP() AS ingest_timestamp 
        FROM delta.`{silver_path}`
    )
    SELECT 
        symbol,
        trade_date,
        close,
        sma_7d,
        sma_30d,

        CASE
            WHEN sma_7d > sma_30d THEN 'bullish'
            ELSE 'bearish'
        END AS sma_signal,
        ingest_timestamp

    FROM sma_data

    """
    return spark.sql(query)

df_moving_averages = build_moving_averages(SILVER_FACT_PRICES_PATH)
df_moving_averages.limit(10).display()

## 4. gold_volatility

Rolling volatility metrics per stock — risk indicators.

**Columns:**
- `symbol`, `trade_date` — from Silver
- `daily_return_pct` — calculated from previous day's close
- `volatility_7d` — standard deviation of daily returns over the last 7 days
- `volatility_30d` — standard deviation of daily returns over the last 30 days
- `ingest_timestamp` — when this row was written

**Business value:**
- Quantify risk for each stock
- Compare volatility regimes between stocks
- Detect volatility spikes (potentially indicating market stress)

In [0]:
def build_gold_volatility(silver_path: str) -> DataFrame:
    """
    Builds Simple Moving Averages (7d and 30d) with bullish/bearish signal.
    
    Returns:
        DataFrame with sma_7d, sma_30d and sma_signal per stock and day.
    """
    query = f"""
    WITH with_prev AS (
        SELECT
        symbol,
        trade_date,
        close,
        LAG(close) OVER (PARTITION BY symbol ORDER BY trade_date) AS prev_close
        FROM delta.`{silver_path}`
    ),
    daily_returns AS (
        SELECT 
            symbol,
            trade_date,
            close,
            prev_close,
            ROUND(((close - prev_close) / prev_close) * 100, 2) AS daily_return_pct
        FROM with_prev
        WHERE prev_close IS NOT NULL
    )
    SELECT
        symbol,
        trade_date,
        close,
        daily_return_pct,
        -- STDDEV em janela de 7 dias
        ROUND(STDDEV(daily_return_pct) OVER (
            PARTITION BY symbol
            ORDER BY trade_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) AS volatility_7d,
        -- STDDEV em janela de 30 dias
        ROUND(STDDEV(daily_return_pct) OVER (
            PARTITION BY symbol
            ORDER BY trade_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 2) AS volatility_30d,
        CURRENT_TIMESTAMP() AS ingest_timestamp
    FROM daily_returns
    """
    return spark.sql(query)

df3 = build_gold_volatility(SILVER_FACT_PRICES_PATH)
df3.limit(10).display()

## 5. gold_stock_comparison

Relative performance ranking across stocks within the dataset.

**Columns:**
- `symbol`, `trade_date` — from Silver
- `close` — closing price
- `start_price` — first available close price per stock
- `return_since_start_pct` — cumulative return % from start
- `rank_by_return` — relative rank vs other stocks for the same day
- `ingest_timestamp` — when this row was written

**Business value:**
- Identify best and worst performers over the period
- Support portfolio construction decisions
- Compare Portuguese vs US stocks on equal footing

In [0]:
def build_gold_stock_comparison(silver_path: str) -> DataFrame:
    """
    Builds Simple Moving Averages (7d and 30d) with bullish/bearish signal.
    
    Returns:
        DataFrame with sma_7d, sma_30d and sma_signal per stock and day.
    """
    query = f"""
        WITH with_start AS (
        SELECT
            symbol,
            trade_date,
            close,
            FIRST_VALUE(close) OVER (PARTITION BY symbol
            ORDER BY trade_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS start_price
        FROM delta.`{silver_path}`
    )
    SELECT
        symbol,
        trade_date,
        close,
        start_price,
        ROUND(
            CASE
                WHEN start_price IS NULL OR start_price = 0 THEN NULL
                ELSE ((close - start_price) / start_price) * 100
            END,
            2
        ) AS return_since_start_pct,

        DENSE_RANK() OVER (
            PARTITION BY trade_date
            ORDER BY ((close - start_price) / start_price) DESC) AS rank_by_return,
        CURRENT_TIMESTAMP() AS ingest_timestamp
    FROM with_start

    """
    return spark.sql(query)

df2 = build_gold_stock_comparison(SILVER_FACT_PRICES_PATH)
df2.limit(10).display()

In [0]:
# SÓ CORRER EM CASO DA CÉLULA SEGUINTE DAR MISMACH - Limpa paths antigos para escrever do zero 
paths_to_clean = [
    GOLD_DAILY_SUMMARY_PATH,
    GOLD_VOLUME_ANALYSIS_PATH,
    GOLD_MOVING_AVERAGES_PATH,
    GOLD_VOLATILITY_PATH,
    GOLD_STOCK_COMPARISON_PATH,
]

for path in paths_to_clean:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"✅ Removed {path}")
    except Exception as e:
        print(f"⚠️ {path} did not exist or failed: {e}")

In [0]:
# Test and write volume analysis
df_volume = build_volume_analysis(SILVER_FACT_PRICES_PATH)
df_volume.show(5)
write_gold_table(df_volume, GOLD_VOLUME_ANALYSIS_PATH, "gold_volume_analysis")

# Test and write moving averages
df_sma = build_moving_averages(SILVER_FACT_PRICES_PATH)
df_sma.show(5)
write_gold_table(df_sma, GOLD_MOVING_AVERAGES_PATH, "gold_moving_averages")

# Test and write volatility
df_volatility = build_gold_volatility(SILVER_FACT_PRICES_PATH)
df_volatility.show(5)
write_gold_table(df_volatility, GOLD_VOLATILITY_PATH, "gold_volatility")

# Test and write stock comparison
df_comparison = build_gold_stock_comparison(SILVER_FACT_PRICES_PATH)
df_comparison.show(5)
write_gold_table(df_comparison, GOLD_STOCK_COMPARISON_PATH, "gold_stock_comparison")

In [0]:
# # Registar tabelas Gold no Unity Catalog para SQL Dashboard
# spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# # Registar cada tabela Gold
# for table, path in [
#     ("daily_summary",    GOLD_DAILY_SUMMARY_PATH),
#     ("moving_averages",  GOLD_MOVING_AVERAGES_PATH),
#     ("volatility",       GOLD_VOLATILITY_PATH),
#     ("volume_analysis",  GOLD_VOLUME_ANALYSIS_PATH),
#     ("stock_comparison", GOLD_STOCK_COMPARISON_PATH),
# ]:
#     spark.sql(f"""
#         CREATE TABLE IF NOT EXISTS gold.{table}
#         USING DELTA
#         LOCATION '{path}'
#     """)
#     print(f"✅ gold.{table} registada")